# データエンコーディング

## はじめにと表記法

量子アルゴリズムを使用するには、古典データを何らかの方法で量子回路に取り込む必要があります。これは通常、データ*エンコーディング*と呼ばれますが、データ*ローディング*と呼ばれることもあります。これまでのレッスンで扱った特徴量マップ (データの特徴量をある空間から別の空間へ写す写像) の概念を思い出してください。古典データを量子コンピューターに転送するだけでも一種の写像であり、特徴量マップと呼ぶことができます。実際には、Qiskit に組み込まれている特徴量マップ (`z_feature_map` や `zz_feature_map` など) には、通常、回転層とエンタングル層が含まれており、状態をヒルベルト空間内の多くの次元へと拡張します。このエンコーディング処理は量子機械学習アルゴリズムの重要な部分であり、その計算能力に直接影響します。

以下で紹介するエンコーディング手法の中には、古典的に効率よくシミュレーションできるものもあります。これは特に、積状態を生成する (つまり量子ビットをエンタングルさせない) エンコーディング手法で顕著です。また、量子の有用性が得られる可能性が最も高いのは、データセットが持つ量子的な複雑さとエンコーディング手法がうまく適合している場合であることを忘れないでください。そのため、最終的には独自のエンコーディング回路を作成することになる可能性が非常に高いでしょう。ここでは、さまざまなエンコーディング戦略を比較・対照し、何が可能かを把握していただくために、多種多様なエンコーディング戦略を紹介します。エンコーディング手法の有用性については、非常に一般的な主張がいくつか可能です。たとえば、完全なエンタングルメント方式を用いた `efficient_su2` (後述) は、積状態を生成する手法 (`z_feature_map` など) よりも、データの量子的な特徴を捉えられる可能性がはるかに高くなります。しかし、だからといって `efficient_su2` が量子スピードアップを得るのに十分である、あるいはデータセットに十分適合しているというわけではありません。そのためには、モデル化や分類の対象となるデータの構造を慎重に検討する必要があります。また、回路の深さとのバランスも考慮しなければなりません。回路内の量子ビットを完全にエンタングルさせる特徴量マップの多くは非常に深い回路となり、現在の量子コンピューターで有用な結果を得るには深すぎるからです。

### 表記法

データセットは $M$ 個のデータベクトルの集合 $\text{X} = \{\vec{x}^{(j)}\,|\,j\in [M]\}$ であり、各ベクトルは $N$ 次元、つまり $\vec{x}^{(j)}=(\vec{x}^{(j)}_1,\ldots,\vec{x}^{(j)}_N)\in\mathbb{R}^N$ です。文脈によっては、ベクトルの成分に $0$ から $N-1$ までの番号が振られている場合もあります。これは複素数のデータ特徴量にも拡張できます。このレッスンでは、集合全体 $(\text{X})$ や、$\vec{x}^{(j)}$ のような特定の要素に対してこれらの表記を使うこともあります。しかし、ほとんどの場合はデータセットから一度に 1 つのベクトルをロードすることについて扱うため、$N$ 個の特徴量を持つ 1 つのベクトルを単に $\vec{x}$ と表記することがよくあります。

さらに、データベクトル $\vec{x}$ の特徴量マップ $\Phi$ を表すのに記号 $\Phi(\vec{x})$ を使うのが一般的です。特に量子コンピューティングでは、写像を $U(\vec{x})$ で表すことが一般的であり、この表記はこれらの操作がユニタリーであることを強調しています。どちらも特徴量マップなので、両方に同じ記号を使っても誤りではありません。このコースでは、主に次のように使い分けます。

* 機械学習における特徴量マップを一般的に論じる場合は $\Phi(\vec{x})$
* 特徴量マップの回路実装を論じる場合は $U(\vec{x})$

### 正規化と情報の損失

古典的な機械学習では、学習データの特徴量を「正規化」またはリスケールすることがよくあり、これによりモデルの性能が向上することが多くあります。一般的な方法の 1 つは、最小-最大正規化 (min-max normalization) や標準化を用いることです。最小-最大正規化では、データ行列 $\text{X}$ の特徴量の列 (たとえば特徴量 $k$) を次のように正規化します。

$$
x^{'(i)}_k = \frac{x^{(i)}_k - \text{min}\{x^{(j)}_k\,|\,\vec{x}^{(j)}\in [\text{X}]\}}{\text{max}\{x^{(j)}_k\,|\,\vec{x}^{(j)}\in [\text{X}]\}-\text{min}\{x^{(j)}_k\,|\,\vec{x}^{(j)}\in [\text{X}]\}}
$$

ここで、min と max は、データセット $\text{X}$ 内の $M$ 個のデータベクトルにわたる特徴量 $k$ の最小値と最大値を表します。これにより、すべての特徴量の値は単位区間に収まります。すなわち、すべての $i\in [M]$、$k\in[N]$ に対して $x^{'(i)}_k \in [0,1]$ となります。

正規化は量子力学や量子コンピューティングにおいても基本的な概念ですが、最小-最大正規化とは少し異なります。量子力学における正規化では、状態ベクトル $|\psi\rangle$ の長さ (量子コンピューティングの文脈では 2-ノルム) が 1 に等しいこと、すなわち $\|\psi\|=\sqrt{\langle\psi|\psi\rangle} = 1$ が要求され、これにより測定確率の総和が 1 になることが保証されます。状態は 2-ノルムで割ることによって、つまり次のようにリスケールすることによって正規化されます。

$$
|\psi\rangle\rightarrow\|\psi\|^{-1}|\psi\rangle
$$

量子コンピューティングや量子力学において、これは人がデータに課す正規化ではなく、量子状態の基本的な性質です。エンコーディング方式によっては、この制約がデータのリスケールの仕方に影響する場合があります。たとえば、振幅エンコーディング (後述) では、量子力学の要請によりデータベクトルが $\vert\vec{x}^{(j)}\vert = 1$ と正規化され、これはエンコードされるデータのスケーリングに影響します。位相エンコーディングでは、量子ビットの位相角へのエンコーディングにおける $2\pi$ を法とする効果によって情報が失われないように、特徴量の値を $\vec{x}^{(j)}_i \in (0,2\pi]$ となるようにリスケールすることが推奨されます[\[1,2\]](#references)。


## エンコーディングの手法

以下のいくつかのセクションでは、$M=5$ 個のデータベクトルからなり、各ベクトルが $N=3$ 個の特徴量を持つ、小さな古典データセットの例 $\text{X}_\text{ex}$ を参照します。

$$
\text{X}_{\text{ex}}=\{(4,8,5),(9,8,6),(2,9,2),(5,7,0),(3,7,5)\}
$$

上で導入した表記を用いると、たとえば集合 $\text{X}_{\text{ex}}$ の $4^\text{th}$ 番目のデータベクトルの $1^\text{st}$ 番目の特徴量は $\vec{x}^{(4)}_1 = 5$ であると言えます。


### 基底エンコーディング

基底エンコーディングは、古典的な $P$ ビットの文字列を $P$ 量子ビット系の計算基底状態にエンコードします。たとえば $\vec{x}^{(1)}_3 = 5 = 0(2^3)+1(2^2)+0(2^1)+1(2^0)$ を考えます。これは $4$ ビットの文字列として $(0101)$ と表すことができ、$4$ 量子ビット系では量子状態 $|0101\rangle$ として表せます。より一般的には、$P$ ビットの文字列 $\vec{x}^{(j)}_k = (b_1, b_2, ... , b_P)$ に対応する $P$ 量子ビットの状態は $|x^{(j)}_k\rangle = | b_1, b_2, ... , b_P \rangle$ (ただし $n = 1 , \dots , P$ に対して $b_n \in \{0,1\}$) となります。これは単一の特徴量に対するものであることに注意してください。

量子コンピューティングにおける基底エンコーディングでは、古典ビットの 1 つ 1 つを別々の量子ビットで表し、データの 2 進表現を計算基底の量子状態に直接マッピングします。複数の特徴量をエンコードする必要がある場合は、まず各特徴量を 2 進形式に変換し、それぞれを別々の量子ビットのグループ (特徴量ごとに 1 グループ) に割り当てます。このとき、各量子ビットはその特徴量の 2 進表現の 1 ビットに対応します。

例として、ベクトル (5, 7, 0) をエンコードしてみましょう。

すべての特徴量が 4 ビットで格納されるとします (必要以上ですが、10 進数で 1 桁の任意の整数を表すには十分です)。

5 → 2 進数 0101

7 → 2 進数 0111

0 → 2 進数 0000

これらのビット列は 4 量子ビットずつの 3 つの組に割り当てられるため、全体の 12 量子ビットの基底状態は次のようになります。

$$
∣0101 0111 0000⟩
$$

ここで、最初の 4 量子ビットが 1 番目の特徴量を、次の 4 量子ビットが 2 番目の特徴量を、最後の 4 量子ビットが 3 番目の特徴量を表します。以下のコードは、データベクトル (5,7,0) を量子状態に変換するもので、他の 1 桁の特徴量にも対応できるように一般化されています。


In [1]:
from qiskit import QuantumCircuit

# エンコードするデータ点
x = 5  # 2 進数: 0101
y = 7  # 2 進数: 0111
z = 0  # 2 進数: 0000

# それぞれを 4 ビットの 2 進数リストに変換
x_bits = [int(b) for b in format(x, "04b")]  # [0,1,0,1]
y_bits = [int(b) for b in format(y, "04b")]  # [0,1,1,1]
z_bits = [int(b) for b in format(z, "04b")]  # [0,0,0,0]

# すべてのビットを結合
all_bits = x_bits + y_bits + z_bits  # [0,1,0,1,0,1,1,1,0,0,0,0]

# 12 量子ビットの量子回路を初期化
qc = QuantumCircuit(12)

# ビットが 1 の位置に X ゲートを適用
for idx, bit in enumerate(all_bits):
    if bit == 1:
        qc.x(idx)

qc.draw("mpl")

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/85ee995f-1e50-4860-a24c-16bbc8b5c8b0-0.avif" alt="Output of the previous code cell" />

#### 理解度チェック

例のデータセット $\text{X}_{\text{ex}}$ の最初のベクトル

$\vec{x}^{(1)}=(4,8,5)$

を、基底エンコーディングを使ってエンコードするコードを書いてください。

<Accordion>
  <AccordionItem title="解答">
    ```python
    import math
    from qiskit import QuantumCircuit

    # エンコードするデータ点
    x = 4  # 2 進数: 0100
    y = 8  # 2 進数: 1000
    z = 5  # 2 進数: 0101

    # それぞれを 4 ビットの 2 進数リストに変換
    x_bits = [int(b) for b in format(x, '04b')]  # [0,1,0,0]
    y_bits = [int(b) for b in format(y, '04b')]  # [1,0,0,0]
    z_bits = [int(b) for b in format(z, '04b')]  # [0,1,0,1]

    # すべてのビットを結合
    all_bits = x_bits + y_bits + z_bits  # [0,1,0,0,1,0,0,0,0,1,0,1]

    # 12 量子ビットの量子回路を初期化
    qc = QuantumCircuit(12)

    # ビットが 1 の位置に X ゲートを適用
    for idx, bit in enumerate(all_bits):
        if bit == 1:
            qc.x(idx)

    qc.draw('mpl')
    ```
  </AccordionItem>
</Accordion>


### 振幅エンコーディング

振幅エンコーディングは、データを量子状態の振幅にエンコードします。正規化された古典的な $N$ 次元データベクトル $\vec{x}^{(j)}$ を、$n$ 量子ビットの量子状態 $|\psi_x\rangle$ の振幅として表現します。

$$
|\psi^{(j)}_x\rangle = \frac{1}{\alpha}\sum_{i=1}^N x^{(j)}_i |i\rangle
$$

ここで、$N$ はこれまでと同じくデータベクトルの次元であり、$\vec{x}^{(j)}_i$ は $\vec{x}^{(j)}$ の $i^{th}$ 番目の要素です。計算基底状態は固定された順序で並んでいるものとし、ここではその順序を $i$ で参照しています。同等に、このエンコーディングを $\sum_{i=0}^{N-1} x_i |i\rangle$ と書くこともできます。ここで $|i\rangle$ は整数 $i$ の 2 進表現でラベル付けされた基底状態を表します。一貫していればどちらでも構いません。ここで $\alpha$ は、エンコードされるデータから決まる正規化定数です。これは量子力学によって課される次の正規化条件です。

$$
\sum_{i=1}^N \left|x^{(j)}_i\right|^2 = \left|\alpha\right|^2.
$$

一般に、これはすべてのデータベクトルにわたって各特徴量に用いる最小/最大正規化とは異なる条件です。これを具体的にどう扱うかは問題によって異なります。しかし、上記の量子力学的な正規化条件を回避する方法はありません。

振幅エンコーディングでは、データベクトル内の各特徴量は、それぞれ異なる量子状態の振幅として格納されます。$n$ 量子ビットの系は $2^n$ 個の振幅を持つため、$N$ 個の特徴量を振幅エンコーディングするには $n \ge \mathrm{log}_2(N)$ 個の量子ビットが必要です。

例として、例のデータセット $\text{X}_\text{ex}$ の最初のベクトル $\vec{x}^{(1)} = (4,8,5)$ を振幅エンコーディングでエンコードしてみましょう。得られるベクトルを正規化すると、次のようになります。

$$
\sum_{i=1}^N \left|x^{(1)}_i\right|^2 = 4^2+8^2+5^2 = 105 = \left|\alpha\right|^2 \rightarrow \alpha = \sqrt{105}
$$

そして、得られる 2 量子ビットの量子状態は次のようになります。

$$
|\psi(\vec{x}^{(1)})\rangle = \frac{1}{\sqrt{105}}(4|00\rangle+8|01\rangle+5|10\rangle+0|11\rangle)
$$

上の例では、ベクトルの特徴量の数 $N=3$ は 2 のべき乗ではありません。$N$ が 2 のべき乗でない場合は、$2^n\geq N$ となるように量子ビット数 $n$ の値を選び、振幅ベクトルを情報を持たない定数 (ここでは 0) で埋めます。

基底エンコーディングと同様に、データセットをエンコードする状態を計算したら、Qiskit では `initialize` 関数を使ってその状態を準備できます。


In [ ]:
import math

desired_state = [
    1 / math.sqrt(105) * 4,
    1 / math.sqrt(105) * 8,
    1 / math.sqrt(105) * 5,
    1 / math.sqrt(105) * 0,
]

qc = QuantumCircuit(2)
qc.initialize(desired_state, [0, 1])

qc.decompose(reps=5).draw(output="mpl")

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/19810c6d-8d60-49ee-bd6f-6f6fbd5e7363-0.avif" alt="Output of the previous code cell" />

振幅エンコーディングの利点は、前述のとおり、エンコードに $\mathrm{log}_2(N)$ 個の量子ビットしか必要としないことです。しかし、後続のアルゴリズムは量子状態の振幅に対して操作を行う必要があり、量子状態を準備・測定する方法は効率的でない傾向があります。


#### 理解度チェック

次のベクトル (例のデータセットの 2 つのベクトルからなる)

$\vec{x}=(9,8,6,2,9,2)$

を振幅エンコーディングでエンコードするための、正規化された状態を書いてください。

<Accordion>
  <AccordionItem title="解答">
    6 つの数値をエンコードするには、その振幅にエンコードできる状態が少なくとも 6 つ必要です。そのためには 3 量子ビットが必要になります。未知の正規化因子 $\alpha$ を用いると、次のように書けます。

    $$
    |\psi\rangle = \frac{1}{\alpha}(9|000\rangle+8|001\rangle+6|010\rangle+2|011\rangle+9|100\rangle+2|101\rangle+0|110\rangle+0|111\rangle)
    $$

    ここで、

    $$
    \langle \psi|\psi\rangle = \frac{1}{|\alpha|^2}\times(9^2+8^2+6^2+2^2+9^2+2^2+0^2+0^2) = \frac{1}{|\alpha|^2}\times(270)=1 \rightarrow \alpha = \sqrt{270}
    $$

    となるので、最終的に次のようになります。

    $$
    |\psi\rangle = \frac{1}{\sqrt{270}}(9|000\rangle+8|001\rangle+6|010\rangle+2|011\rangle+9|100\rangle+2|101\rangle+0|110\rangle+0|111\rangle)
    $$
  </AccordionItem>
</Accordion>

同じデータベクトル $\vec{x}=(9,8,6,2,9,2)$ について、振幅エンコーディングを使ってこれらのデータ特徴量をロードする回路を作成するコードを書いてください。

<Accordion>
  <AccordionItem title="解答">
    ```python
    desired_state = [
        9 / math.sqrt(270),
        8 / math.sqrt(270),
        6 / math.sqrt(270),
        2 / math.sqrt(270),
        9 / math.sqrt(270),
        2 / math.sqrt(270),
        0,
        0,
    ]

    print(desired_state)

    qc = QuantumCircuit(3)
    qc.initialize(desired_state, [0, 1, 2])
    qc.decompose(reps=8).draw(output="mpl")
    ```

    \[0.5477225575051662, 0.48686449556014766, 0.36514837167011077, 0.12171612389003691, 0.5477225575051662, 0.12171612389003691, 0, 0]

    !["前のコードセルの出力"](https://quantum.cloud.ibm.com/learning/images/courses/quantum-machine-learning/data-encoding/checkin2.avif)
  </AccordionItem>
</Accordion>

非常に大きなデータベクトルを扱う必要がある場合もあります。次のベクトルを考えてください。

$\vec{x}=(4,8,5,9,8,6,2,9,2,5,7,0,3,7,5).$

正規化を自動化し、振幅エンコーディングのための量子回路を生成するコードを書いてください。

<Accordion>
  <AccordionItem title="解答">
    解答は多数考えられます。以下は、途中のいくつかのステップを出力するコードです。

    ```python
    import numpy as np
    from math import sqrt

    init_list = [4, 8, 5, 9, 8, 6, 2, 9, 2, 5, 7, 0, 3, 7, 5]
    qubits = round(np.log(len(init_list)) / np.log(2) + 0.4999999999)
    need_length = 2**qubits
    pad = need_length - len(init_list)
    for i in range(0, pad):
        init_list.append(0)

    init_array = np.array(init_list)  # 正規化前のデータベクトル
    length = sqrt(
        sum(init_array[i] ** 2 for i in range(0, len(init_array)))
    )  # ベクトルの長さ
    norm_array = init_array / length  # 正規化された配列
    print("Normalized array:")
    print(norm_array)
    print()

    qubit_numbers = []
    for i in range(0, qubits):
        qubit_numbers.append(i)
    print(qubit_numbers)

    qc = QuantumCircuit(qubits)
    qc.initialize(norm_array, qubit_numbers)
    qc.decompose(reps=7).draw(output="mpl")
    ```

    Normalized array:
    \[0.17342199 0.34684399 0.21677749 0.39019949 0.34684399 0.26013299
    0.086711   0.39019949 0.086711   0.21677749 0.30348849 0.
    0.1300665  0.30348849 0.21677749 0.        ]

    \[0, 1, 2, 3]

    !["前のコードセルの出力"](https://quantum.cloud.ibm.com/learning/images/courses/quantum-machine-learning/data-encoding/checkin3.avif)
  </AccordionItem>
</Accordion>

振幅エンコーディングには、基底エンコーディングと比べて利点があると思いますか? もしあれば、説明してください。

<Accordion>
  <AccordionItem title="解答">
    いくつかの答えが考えられます。1 つの答えは、基底状態の順序が固定されているため、この振幅エンコーディングではエンコードされた数値の順序が保たれるということです。また、多くの場合、より高密度にエンコードされます。
  </AccordionItem>
</Accordion>


振幅エンコーディングの利点は、$N$ 次元 ($N$ 個の特徴量) のデータベクトル $\vec{x}\rightarrow|\vec{x}\rangle$ に対して $\log_2(N)$ 個の量子ビットしか必要としないことです。しかし、振幅エンコーディングは一般に、任意の状態準備を必要とする非効率な手順であり、CNOT ゲートの数が指数関数的になります。言い換えると、状態準備の実行時間の計算量は次元数に対して $\mathcal O(N)$ の多項式オーダーとなります。ここで $N = 2^n$ であり、$n$ は量子ビット数です。振幅エンコーディングは「時間の指数関数的な増大を代償として、空間の指数関数的な節約をもたらす」[\[3\]](#references) ものですが、特定のケースでは実行時間を $\mathcal O(\log N)$ に抑えることが可能です[\[4\]](#references)。エンドツーエンドで量子スピードアップを実現するには、データローディングの実行時間の計算量を考慮する必要があります。


### 角度エンコーディング

角度エンコーディングは、量子サポートベクターマシン (QSVM) や変分量子回路 (VQC) など、パウリ特徴量マップを使用する多くの QML モデルで注目されています。角度エンコーディングは、後述する位相エンコーディングや密角度エンコーディングと密接に関連しています。ここでは「角度エンコーディング」を、$\theta$ 方向の回転、つまり、たとえば $R_X$ ゲートや $R_Y$ ゲートによって実現される $z$ 軸から離れる方向への回転を指すものとして使います[\[1,3\]](#references)。実際には、データは*任意の*回転や回転の組み合わせにエンコードできます。しかし、文献では $R_Y$ が一般的なので、ここではそれを重点的に扱います。

単一の量子ビットに適用した場合、角度エンコーディングはデータの値に比例した Y 軸周りの回転を与えます。データセット内の $j^\text{th}$ 番目のデータベクトルの単一の ($k^\text{th}$ 番目の) 特徴量 $\vec{x}^{(j)}_k$ のエンコーディングを考えます。

$$
|\vec{x}^{(j)}_k\rangle = R_Y(\theta=\vec{x}^{(j)}_k)|0\rangle = \textstyle\cos\left(\frac{\vec{x}^{(j)}_k}{2}\right)|0\rangle + \sin\left(\frac{\vec{x}^{(j)}_k}{2}\right)|1\rangle.
$$

あるいは、$R_X(\theta)$ ゲートを使って角度エンコーディングを行うこともできますが、その場合、エンコードされた状態は $R_Y(\theta)$ の場合と比べて複素数の相対位相を持つことになります。

角度エンコーディングは、これまでに説明した 2 つの手法とはいくつかの点で異なります。角度エンコーディングでは、次のようになります。

* 各特徴量の値は対応する量子ビットにマッピングされ ($\vec{x}^{(j)}_k \rightarrow Q_k$)、量子ビットは積状態のままとなります。
* データ点の特徴量の集合全体ではなく、一度に 1 つの数値がエンコードされます。
* $N$ 個のデータ特徴量に対して $n$ 個の量子ビットが必要です (ただし $n\leq N$)。ここでは多くの場合、等号が成り立ちます。$n<N$ がどのようにして可能になるかは、以降のいくつかのセクションで説明します。
* 得られる回路の深さは一定です (トランスパイル前の深さは通常 1 です)。

回路の深さが一定であるため、現在の量子ハードウェアに特に適しています。データを $\theta$ を用いてエンコードすること (特に、Y 軸角度エンコーディングを選択したこと) のもう 1 つの特徴は、特定の応用で役立つ実数値の量子状態が生成されることです。Y 軸回転の場合、データは実数値の角度 $\theta \in (0, 2\pi]$ による Y 軸回転ゲート $R_Y(\theta)$ でマッピングされます ([Qiskit RYGate](/docs/api/qiskit/qiskit.circuit.library.RYGate))。位相エンコーディング (後述) と同様に、情報の損失やその他の望ましくない影響を防ぐため、$\vec{x}^{(j)}_k \in (0,2\pi]$ となるようにデータをリスケールすることをお勧めします。

次の Qiskit コードは、単一の量子ビットを初期状態 $|0\rangle$ から回転させて、データ値 $\vec{x}^{(j)}_k=\frac{1}{2}\pi$ をエンコードします。


In [38]:
from qiskit.quantum_info import Statevector
from math import pi

qc = QuantumCircuit(1)
state1 = Statevector.from_instruction(qc)
qc.ry(pi / 2, 0)  # 位相ゲートが角度 pi/2 だけ回転させる
state2 = Statevector.from_instruction(qc)
states = state1, state2

状態ベクトルに対する作用を可視化する関数を定義します。関数定義の詳細は重要ではありませんが、状態ベクトルとその変化を可視化できることは重要です。


In [ ]:
import numpy as np
from qiskit.visualization.bloch import Bloch
from qiskit.visualization.state_visualization import _bloch_multivector_data


def plot_Nstates(states, axis, plot_trace_points=True):
    """この関数は N 個の状態を 1 つのブロッホ球にプロットします"""
    bloch_vecs = [_bloch_multivector_data(s)[0] for s in states]

    if axis is None:
        bloch_plot = Bloch()
    else:
        bloch_plot = Bloch(axes=axis)

    bloch_plot.add_vectors(bloch_vecs)

    if len(states) > 1:

        def rgba_map(x, num):
            g = (0.95 - 0.05) / (num - 1)
            i = 0.95 - g * num
            y = g * x + i
            return (0.0, y, 0.0, 0.7)

        num = len(states)
        bloch_plot.vector_color = [rgba_map(x, num) for x in range(1, num + 1)]

    bloch_plot.vector_width = 3
    bloch_plot.vector_style = "simple"

    if plot_trace_points:

        def trace_points(bloch_vec1, bloch_vec2):
            # bloch_vec = (x,y,z)
            n_points = 15
            thetas = np.arccos([bloch_vec1[2], bloch_vec2[2]])
            phis = np.arctan2(
                [bloch_vec1[1], bloch_vec2[1]], [bloch_vec1[0], bloch_vec2[0]]
            )
            if phis[1] < 0:
                phis[1] = phis[1] + 2 * pi
            angles0 = np.linspace(phis[0], phis[1], n_points)
            angles1 = np.linspace(thetas[0], thetas[1], n_points)

            xp = np.cos(angles0) * np.sin(angles1)
            yp = np.sin(angles0) * np.sin(angles1)
            zp = np.cos(angles1)
            pnts = [xp, yp, zp]
            bloch_plot.add_points(pnts)
            bloch_plot.point_color = "k"
            bloch_plot.point_size = [4] * len(bloch_plot.points)
            bloch_plot.point_marker = ["o"]

        for i in range(len(bloch_vecs) - 1):
            trace_points(bloch_vecs[i], bloch_vecs[i + 1])

    bloch_plot.sphere_alpha = 0.05
    bloch_plot.frame_alpha = 0.15
    bloch_plot.figsize = [4, 4]

    bloch_plot.render()


plot_Nstates(states, axis=None, plot_trace_points=True)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/666700f7-7798-43ce-a8ca-d91e48adda4f-0.avif" alt="Output of the previous code cell" />

ここまでは、単一のデータベクトルの単一の特徴量だけでした。$N$ 個の特徴量を $n$ 個の量子ビットの回転角にエンコードする場合、たとえば $j^\text{th}$ 番目のデータベクトル $\vec{x}^{(j)} = (x_1,...,x_N)$ について、エンコードされた積状態は次のようになります。

$$
|\vec{x}^{(j)}\rangle = \bigotimes^N_{k=1} \cos\left(\frac{\vec{x}^{(j)}_k}{2}\right)|0\rangle + \sin\left(\frac{\vec{x}^{(j)}_k}{2}\right)|1\rangle
$$

あるいは、同等に次のようにも書けます。

$$
|\vec{x}^{(j)}\rangle = \bigotimes^N_{k=1} R_Y(\vec{x}^{(j)}_k)|0\rangle.
$$

文献によっては、正弦/余弦関数の引数を $\vec{x}^{(j)}_k/2$ ではなく $\vec{x}^{(j)}_k$ とするようにエンコーディングを定義している場合があります。これは表記上の好みであり、$R_Y(2 \theta)$ を使うことと同等です。特に、いずれにせよデータは $(0,2\pi]$ の区間にリスケールすべきなので、通常は重要な違いではありません。

#### 理解度チェック

上で説明した角度エンコーディングを使って、データベクトル $\vec{x} = (0, \pi/4, \pi/2)$ をエンコードしてください。

<Accordion>
  <AccordionItem title="解答">
    ```python
    qc = QuantumCircuit(3)
    qc.ry(0, 0)
    qc.ry(math.pi / 4, 1)
    qc.ry(math.pi / 2, 2)
    qc.draw(output="mpl")
    ```

    !["前のコードセルの出力"](https://quantum.cloud.ibm.com/learning/images/courses/quantum-machine-learning/data-encoding/checkin4.avif)
  </AccordionItem>
</Accordion>

上で説明した角度エンコーディングを使う場合、5 つの特徴量をエンコードするには何個の量子ビットが必要ですか?

<Accordion>
  <AccordionItem title="解答">
    5
  </AccordionItem>
</Accordion>


### 位相エンコーディング

位相エンコーディングは、上で説明した角度エンコーディングとよく似ています。量子ビットの位相角とは、$z$ 軸周りに +$x$ 軸から測った実数値の角度 $\phi$ のことです。データは位相回転 $P(\phi) = e^{i\phi/2}R_Z(\phi)$ (ただし $\phi \in (0,2\pi]$) によってマッピングされます (詳しくは [Qiskit PhaseGate](/docs/api/qiskit/qiskit.circuit.library.PhaseGate) を参照してください)。$\vec{x}^{(j)}_k \in (0,2\pi]$ となるようにデータをリスケールすることが推奨されます。これにより、情報の損失やその他の望ましくない可能性のある影響を防ぐことができます[\[1,2\]](#references)。

量子ビットは多くの場合、状態 $|0\rangle$ に初期化されます。これは位相回転演算子の固有状態であるため、位相エンコーディングを実装するには、まず量子ビットの状態を回転させる必要があります。そのため、アダマールゲートで状態を初期化するのが理にかなっています: $H|0\rangle = |+\rangle = \textstyle\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$。単一の量子ビットに対する位相エンコーディングとは、データの値に比例した相対位相を与えることを意味します。

$$
|\vec{x}^{(j)}_k\rangle = P(\phi=\vec{x}^{(j)}_k)|+\rangle = \textstyle\frac{1}{\sqrt{2}}\big(|0\rangle + e^{i\vec{x}^{(j)}_k}|1\rangle\big).
$$

位相エンコーディングの手順では、各特徴量の値を対応する量子ビットの位相にマッピングします ($\vec{x}^{(j)}_k \rightarrow Q_k$)。全体として、位相エンコーディングの回路の深さはアダマール層を含めて 2 であり、効率的なエンコーディング方式となっています。位相エンコーディングされた多量子ビット状態 ($N=n$ 個の特徴量に対して $n$ 量子ビット) は、次のような積状態です。

$$
|\vec{x}^{(j)}\rangle = \bigotimes_{k=1}^{N} P_k(\phi = \vec{x}^{(j)}_k)|+\rangle^{\otimes N} = {\textstyle\frac{1}{\sqrt{2^N}}} \bigotimes_{k=1}^{N}\big(|0\rangle + e^{i\vec{x}^{(j)}_k}|1\rangle\big).
$$

次の Qiskit コードは、まずアダマールゲートで回転させて単一の量子ビットの初期状態を準備し、次に位相ゲートで再び回転させてデータ特徴量 $\vec{x}^{(j)}_k=\frac{1}{2}\pi$ をエンコードします。


In [41]:
qc = QuantumCircuit(1)
qc.h(0)  # アダマールゲートが状態をブロッホ球の赤道まで回転させる
state1 = Statevector.from_instruction(qc)

qc.p(pi / 2, 0)  # 位相ゲートが角度 pi/2 だけ回転させる
state2 = Statevector.from_instruction(qc)

states = state1, state2

qc.draw("mpl", scale=1)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/ba0886eb-1c56-4b15-a731-d94d805254e1-0.avif" alt="Output of the previous code cell" />

定義した plot\_Nstates 関数を使って、$\phi$ 方向の回転を可視化できます。


In [42]:
plot_Nstates(states, axis=None, plot_trace_points=True)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/f7c9cf29-2ad6-43af-a7e3-e590e41d7e67-0.avif" alt="Output of the previous code cell" />

ブロッホ球のプロットは、$\vec{x}^{(j)}_k=\frac{1}{2}\pi$ の場合の Z 軸周りの回転 $|+\rangle \rightarrow P(\frac{1}{2}\pi)|+\rangle$ を示しています。薄い緑色の矢印が最終状態を示しています。

位相エンコーディングは、多くの量子特徴量マップ、特に $Z$ 特徴量マップや $ZZ$ 特徴量マップ、一般的なパウリ特徴量マップなどで使われています。


#### 理解度チェック

上で説明した位相エンコーディングを使って 8 つの特徴量を格納するには、何個の量子ビットが必要ですか?

<Accordion>
  <AccordionItem title="解答">
    8
  </AccordionItem>
</Accordion>

ベクトル $\vec{x}^{(1)}=(4,8,5,9,8,6,2,9,2,5,7,0)$ を位相エンコーディングでエンコードするコードを書いてください。

<Accordion>
  <AccordionItem title="解答">
    解答は多数考えられます。以下はその一例です。

    ```python
    phase_data = [4, 8, 5, 9, 8, 6, 2, 9, 2, 5, 7, 0]
    qc = QuantumCircuit(len(phase_data))
    for i in range(0, len(phase_data)):
        qc.h(i)
        qc.rz(phase_data[i] * 2 * math.pi / float(max(phase_data)), i)
    qc.draw(output="mpl")
    ```

    !["前のコードセルの出力"](https://quantum.cloud.ibm.com/learning/images/courses/quantum-machine-learning/data-encoding/checkin5.avif)
  </AccordionItem>
</Accordion>


### 密角度エンコーディング

密角度エンコーディング (DAE) は、角度エンコーディングと位相エンコーディングを組み合わせたものです。DAE では、2 つの特徴量の値を単一の量子ビットにエンコードできます。一方は Y 軸周りの回転角として、もう一方は $z$ 軸周りの回転角としてエンコードします: $\vec{x}^{(j)}_k$, $\vec{x}^{(j)}_\ell \rightarrow \theta, \phi$。2 つの特徴量は次のようにエンコードされます。

$$
|\vec{x}^{(j)}_k,\vec{x}^{(j)}_\ell\rangle = R_Z(\phi=\vec{x}^{(j)}_\ell) R_Y(\theta=\vec{x}^{(j)}_k)|0\rangle = \cos\left(\frac{\vec{x}^{(j)}_k}{2}\right)|0\rangle + e^{i\vec{x}^{(j)}_\ell} \sin\left(\frac{\vec{x}^{(j)}_k}{2}\right)|1\rangle.
$$

2 つのデータ特徴量を 1 つの量子ビットにエンコードすることで、エンコーディングに必要な量子ビット数が半分 ($2\times$ の削減) になります。これをより多くの特徴量に拡張すると、データベクトル $\vec{x} = (x_1,...,x_N)$ は次のようにエンコードできます。

$$
|\vec{x}\rangle = \bigotimes_{k=1}^{N/2} \cos\left(\frac{x_{2k-1}}{2}\right)|0\rangle + e^{i x_{2k}}\sin\left(\frac{x_{2k-1}}{2}\right)|1\rangle
$$

DAE は、ここで使った正弦関数の代わりに、2 つの特徴量の任意の関数へと一般化できます。これは一般量子ビットエンコーディング (general qubit encoding) と呼ばれます[\[7\]](#references)。

DAE の例として、以下のコードでは特徴量 $x_1=\theta = 3\pi/8$ と $x_2=\phi = 7\pi/4$ をエンコードし、そのエンコーディングを可視化します。


In [44]:
qc = QuantumCircuit(1)
state1 = Statevector.from_instruction(qc)
qc.ry(3 * pi / 8, 0)
state2 = Statevector.from_instruction(qc)
qc.rz(7 * pi / 4, 0)
state3 = Statevector.from_instruction(qc)
states = state1, state2, state3

plot_Nstates(states, axis=None, plot_trace_points=True)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/9a6bb041-d7a1-4e29-a463-81b93b900e96-0.avif" alt="Output of the previous code cell" />

#### 理解度チェック

上の説明に基づくと、密エンコーディングを使って 6 つの特徴量をエンコードするには何個の量子ビットが必要ですか?

<Accordion>
  <AccordionItem title="解答">
    3
  </AccordionItem>
</Accordion>

密角度エンコーディングを使ってベクトル $\vec{x}^{(1)}=(4,8,5,9,8,6,2,9,2,5,7,0,3,7,5)$ をロードするコードを書いてください。

<Accordion>
  <AccordionItem title="解答">
    エンコーディング方式の中で使われないパラメーターが 1 つだけ残るという問題を避けるため、リストを「0」で埋めていることに注意してください。

    ```python
    dense_data = [4, 8, 5, 9, 8, 6, 2, 9, 2, 5, 7, 0, 3, 7, 5, 0]
    qc = QuantumCircuit(int(len(dense_data) / 2))
    entry = 0
    for i in range(0, int(len(dense_data) / 2)):
        qc.ry(dense_data[entry] * 2 * math.pi / float(max(dense_data)), i)
        entry = entry + 1
        qc.rz(dense_data[entry] * 2 * math.pi / float(max(dense_data)), i)
        entry = entry + 1
    qc.draw(output="mpl")
    ```

    !["前のコードセルの出力"](https://quantum.cloud.ibm.com/learning/images/courses/quantum-machine-learning/data-encoding/checkin6.avif)
  </AccordionItem>
</Accordion>


## 組み込みの特徴量マップによるエンコーディング

### 任意の位置でのエンコーディング

角度エンコーディング、位相エンコーディング、密エンコーディングでは、各量子ビットに 1 つの特徴量 (または量子ビットあたり 2 つの特徴量) をエンコードした積状態を準備しました。これは基底エンコーディングや振幅エンコーディングとは異なります。これらの手法はエンタングルした状態を利用しており、データ特徴量と量子ビットが 1 対 1 に対応していません。たとえば振幅エンコーディングでは、ある特徴量を状態 $|01\rangle$ の振幅とし、別の特徴量を $|10\rangle$ の振幅とすることがあります。一般に、積状態にエンコードする手法では回路が浅くなり、各量子ビットに 1 つまたは 2 つの特徴量を格納できます。エンタングルメントを利用し、特徴量を量子ビットではなく状態に対応付ける手法では、回路が深くなりますが、平均して量子ビットあたりにより多くの特徴量を格納できます。

しかし、エンコーディングは完全に積状態で行う必要も、振幅エンコーディングのように完全にエンタングルした状態で行う必要もありません。実際、Qiskit に組み込まれている多くのエンコーディング方式では、最初だけでなく、エンタングル層の前後の両方でエンコードできます。これは「データの再アップロード (data reuploading)」として知られています。関連する研究については、参考文献 \[5] および \[6] を参照してください。

このセクションでは、組み込みのエンコーディング方式をいくつか使い、可視化します。このセクションのすべての手法は、$N$ 個の特徴量を、$n$ 個の量子ビット ($n \leq N$) 上の $N$ 個のパラメーター化ゲートの回転としてエンコードします。与えられた量子ビット数でデータのロード量を最大化することだけが考慮すべき点ではないことに注意してください。多くの場合、量子ビット数よりも回路の深さのほうがさらに重要な考慮事項となることがあります。


### Efficient SU2

エンタングルメントを伴うエンコーディングの一般的で有用な例として、Qiskit の [`efficient_su2`](/docs/api/qiskit/qiskit.circuit.library.EfficientSU2) 回路があります。驚くべきことに、この回路では、たとえばわずか 2 量子ビットで 8 つの特徴量をエンコードできます。実際に確認してから、それがなぜ可能なのかを理解してみましょう。


In [46]:
from qiskit.circuit.library import efficient_su2

circuit = efficient_su2(num_qubits=2, reps=1, insert_barriers=True)
circuit.decompose().draw(output="mpl")

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/6b657226-ae95-41f6-b78b-5def930d0080-0.avif" alt="Output of the previous code cell" />

状態を書き下す際には、$|q_2,q_1,q_0\rangle$ や $|q_2\rangle\otimes|q_1\rangle\otimes|q_0\rangle$ のように、最下位の量子ビットを一番右に並べる Qiskit の規約を使います。これらの状態はすぐに非常に複雑になるため、この珍しい例は、そうした状態が明示的に書き下されることがめったにない理由を説明する助けになるかもしれません。

系は状態 $|00\rangle$ から始まります。最初のバリア ($b1$ とラベル付けする位置) までの状態は次のとおりです。

$$
|\psi\rangle_{b1} = \left(\cos\left(\frac{\theta_1}{2}\right)|0\rangle+\sin\left(\frac{\theta_1}{2}\right)e^{i\theta_3}|1\rangle\right)\otimes\left(\cos\left(\frac{\theta_0}{2}\right)|0\rangle+\sin\left(\frac{\theta_0}{2}\right)e^{i\theta_2}|1\rangle\right)
$$

これは以前に見た密エンコーディングそのものです。次に、CNOT ゲートの後、2 番目のバリア ($b2$) での状態は次のようになります。


$$
\begin{aligned}
|\psi\rangle_{b2} = & \cos\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_0}{2}\right)|00\rangle+\cos\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_0}{2}\right)e^{i\theta_2}|11\rangle\\
+ & \sin\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_0}{2}\right)e^{i\theta_3}|10\rangle+\sin\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_0}{2}\right)e^{i\theta_2}e^{i\theta_3}|01\rangle
\end{aligned}
$$



ここで最後の単一量子ビット回転の組を適用し、同じ状態の項をまとめると、次のようになります。


$$
\begin{align*}
|\psi\rangle_{\text{final}} = &
\left[\cos\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)-\sin\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\cos\left(\frac{\theta_4}{2}\right)\right.\\

+ & \left.\sin\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)-\sin\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\sin\left(\frac{\theta_4}{2}\right)e^{i\theta_2}\right]
|00\rangle\\

+ & \left[\cos\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)-\sin\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\sin\left(\frac{\theta_4}{2}\right)\right.\\

+ & \left.\sin\left(\frac{\theta_0}{2}\right)\left(-\cos\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)+\sin\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\cos\left(\frac{\theta_4}{2}\right)e^{i\theta_2}\right]
e^{i\theta_6}|01\rangle\\

+ & \left[\cos\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)+\sin\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\cos\left(\frac{\theta_4}{2}\right)\right.\\

- & \left.\sin\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)+\sin\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\sin\left(\frac{\theta_4}{2}\right)e^{i\theta_2}\right]
e^{i\theta_7}|10\rangle\\

+ & \left[\cos\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)+\sin\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\sin\left(\frac{\theta_4}{2}\right)\right.\\

+ & \left.\sin\left(\frac{\theta_0}{2}\right)\left(\cos\left(\frac{\theta_1}{2}\right)\cos\left(\frac{\theta_5}{2}\right)+\sin\left(\frac{\theta_1}{2}\right)\sin\left(\frac{\theta_5}{2}\right)e^{i\theta_3}\right)\cos\left(\frac{\theta_4}{2}\right)e^{i\theta_2}\right]
e^{i\theta_6}e^{i\theta_7}|11\rangle

\end{align*}
$$



これを読み解くのはおそらく複雑すぎるでしょう。代わりに、一歩引いて、状態にいくつのパラメーターをロードしたかを考えてみてください。8 つです。しかし、計算基底状態は 4 つしかありません。最終状態は $\psi_\text{final} = c_0|00\rangle+c_1|01\rangle+c_2|10\rangle+c_3|11\rangle$ と書けるので、一見すると、理にかなう数よりも多くのパラメーターをロードしたように見えるかもしれません。しかし、各係数は複素数であることに注意してください! 次のように書くと、

$$
\psi_\text{final} = (a_0+ib_0)|00\rangle+(a_1+ib_1)|01\rangle+(a_2+ib_2)|10\rangle+(a_3+ib_3)|11\rangle
$$

8 つの特徴量をエンコードするための 8 つのパラメーターが、確かに状態上に存在することがわかります。

量子ビット数を増やし、エンタングル層と回転層の繰り返し回数を増やすことで、はるかに多くのデータをエンコードできます。波動関数を書き下すことはすぐに手に負えなくなりますが、それでもエンコーディングが機能する様子を見ることはできます。


ここでは、12 個の特徴量を持つデータベクトル $\vec{x}$ を、3 量子ビットの `efficient_su2` 回路にエンコードします。各パラメーター化ゲートを使って、それぞれ異なる特徴量をエンコードします。

$$
\vec{x} = (0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2)
$$

このデータベクトルでは、特徴量が特定の順序で示されています。それだけを見れば、この順序でエンコードしても逆順でエンコードしても問題ありません。重要なのは、順序を把握し、一貫性を保つことです。回路図を見ると、`efficient_su2` が特定のエンコーディング順序を前提としていることに注意してください。具体的には、パラメーター化ゲートの最初の層を量子ビット 0 から量子ビット 2 まで埋めてから、次の層に移ります。これはリトルエンディアン表記と整合しているとも整合していないとも言えません。というのも、ここではエンコーディング回路が指定される前に、データ特徴量を*先験的に*量子ビットで順序付けることはできないからです。


In [47]:
x = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
circuit = efficient_su2(num_qubits=3, reps=1, insert_barriers=True)
encode = circuit.assign_parameters(x)
encode.decompose().draw(output="mpl")

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/73fc00fe-b98f-4d63-a327-54958a8f5498-0.avif" alt="Output of the previous code cell" />

量子ビット数を増やす代わりに、エンタングル層と回転層の繰り返し回数を増やすことを選ぶこともできます。しかし、有用な繰り返し回数には限界があります。


前述のとおり、トレードオフがあります。より多くの量子ビットや、エンタングル層と回転層のより多くの繰り返しを持つ回路は、より多くのパラメーターを格納できますが、その分回路の深さが大きくなります。いくつかの組み込み特徴量マップの深さについては、後ほど改めて取り上げます。


次に紹介する Qiskit 組み込みのエンコーディング手法は、名前に「feature map (特徴量マップ)」が含まれています。改めて強調しておくと、データを量子回路にエンコードすることは、データを新しい空間 (関係する量子ビットのヒルベルト空間) に移すという意味で、特徴量マッピング*そのもの*です。元の特徴量空間の次元とヒルベルト空間の次元との関係は、エンコーディングに使用する回路によって異なります。


### $Z$ 特徴量マップ

$Z$ 特徴量マップ (ZFM) は、位相エンコーディングの自然な拡張と解釈できます。ZFM は、単一量子ビットゲートの層、すなわちアダマールゲート層と位相ゲート層が交互に並んだもので構成されます。データベクトル $\vec{x}$ が $N$ 個の特徴量を持つとします。特徴量マッピングを行う量子回路は、初期状態に作用するユニタリー演算子として次のように表されます。

$$
\mathscr{U}_{\text{ZFM}}(\vec{x})|0\rangle^{\otimes N}=|\phi(\vec{x})\rangle
$$

ここで、$|0\rangle^{\otimes N}$ は $N$ 量子ビットの基底状態です。この表記は、参考文献 [\[4\]](#references) Havlicek et al. との一貫性のために使用しています。データ特徴量 $x_i$ は、対応する量子ビットに 1 対 1 でマッピングされます。たとえば、データベクトルに 8 つの特徴量がある場合は、8 量子ビットを使用します。ZFM 回路は、アダマールゲート層と位相ゲート層からなるサブ回路を $r$ 回繰り返したもので構成されます。アダマール層は、アルゴリズムの同じ段階において、$n$ 量子ビットのレジスター内のすべての量子ビットに作用するアダマールゲート $H \otimes H \otimes \dots \otimes H = H^{\otimes n}$ からなります。この説明は、$i^\text{th}$ 番目の量子ビットに $P(\vec{x}_i)$ が作用する位相ゲート層にも当てはまります。各 $P$ ゲートは 1 つの特徴量を引数に取りますが、位相ゲート層 ($P(\vec{x}_1)\otimes\ldots P(\vec{x}_k)\otimes\ldots P(\vec{x}_N)$) はデータベクトルの関数です。繰り返し 1 回の ZFM 回路全体のユニタリーは次のとおりです。

$$
\mathscr{U}_{\text{ZFM}}=\big(P(\vec{x}_1)\otimes\ldots P(\vec{x}_k)\otimes\ldots P(\vec{x}_N)H^{\otimes N}\big)=\left(\bigotimes_{k = 1}^N P(\vec{x}_k)\right)H^{\otimes N}
$$

すると、このユニタリーを $r$ 回繰り返したものは次のようになります。

$$
\mathscr{U}^{(r)}_{\text{ZFM}}\left(\vec{x}\right)=\prod_{s=1}^{r}\left[\left(\bigotimes_{k = 1}^N P(\vec{x}_k)\right)H^{\otimes N}\right]
$$

データ特徴量 $x_k$ は、$r$ 回の繰り返しのすべてにおいて同じ方法で位相ゲートにマッピングされます。ZFM の特徴量マップ状態は積状態であり、古典的に効率よくシミュレーションできます[\[4\]](#references)。

まず小さな例として、2 量子ビットの ZFM 回路を Qiskit でコーディングし、描画して単純な回路構造を示します。この例では、データベクトル $\vec{x} = \left(\textstyle\frac{1}{2}\pi, \textstyle\frac{1}{3}\pi\right)$ を用いて、繰り返し 1 回 ($r=1$) を実装します。これは Python におけるベクトルの標準的な順序で書かれており、$0^\text{th}$ 番目の要素が $\textstyle\frac{1}{2}\pi$ であることに注意してください。この $0^\text{th}$ 番目の特徴量は、$0^\text{th}$ 番目の量子ビットにエンコードしても、$N^\text{th}$ 番目の量子ビットにエンコードしても構いません。繰り返しになりますが、特徴量マップによって各量子ビットにエンコードされる特徴量の数が異なるため、特徴量の順序から量子ビットの順序への単一の 1 対 1 の対応が常に存在するとは限りません。ここでも重要なのは、各特徴量がどこにエンコードされているかを把握しておくことです。$Z$ 特徴量マップにパラメーターのリストを与えると、リストの特徴量 0 を、パラメーター化ゲートによって最下位の量子ビット、つまり量子ビット 0 にエンコードします。そのため、手作業で行う場合もこの規約に従います。$\textstyle\frac{1}{2}\pi$ を $0^\text{th}$ 番目の量子ビットに、$\textstyle\frac{1}{3}\pi$ を $1^\text{st}$ 番目の量子ビットにエンコードします。

ZFM 回路のユニタリー演算子は、初期状態に次のように作用します。

$$
\mathscr{U}_{\text{ZFM}}(\bar{x})|00\rangle = P(\bar{x})^{\otimes 2} H^{\otimes 2}|00\rangle = \left( P\left(\textstyle\frac{1}{3}\pi\right)H|0\rangle \right) \otimes \left(P\left(\textstyle\frac{1}{2}\pi\right)H|0\rangle\right).
$$

各量子ビットに対する操作を強調するため、テンソル積の周りで式を並べ替えています。次の Qiskit コードは、アダマールゲートと位相ゲートを明示的に使って ZFM の構造を示します。


In [48]:
qc0 = QuantumCircuit(1)
qc1 = QuantumCircuit(1)

qc0.h(0)
qc0.p(pi / 2, 0)

qc1.h(0)
qc1.p(pi / 3, 0)

# 回路 qc0 と qc1 を 1 つの回路に結合
qc = QuantumCircuit(2)
qc.compose(qc0, [0], inplace=True)
qc.compose(qc1, [1], inplace=True)

qc.draw("mpl", scale=1)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/f5c70df4-faea-4817-a870-95638eb97dbd-0.avif" alt="Output of the previous code cell" />

次に、同じデータベクトル $\vec{x} = \left(\textstyle\frac{1}{2}\pi, \textstyle\frac{1}{3}\pi\right)$ を、Qiskit の [`z_feature_map`](/docs/api/qiskit/qiskit.circuit.library.ZFeatureMap) クラスを使って、繰り返し 3 回 ($r=3$) の ZFM 回路にエンコードします。これにより、全体として量子特徴量マップ $\mathscr{U}_{\text{ZFM}}(\vec{x})$ が得られます。`z_feature_map` クラスでは、デフォルトで、パラメーター $\beta$ は位相ゲートにマッピングされる前に 2 倍されます ($\beta \rightarrow P(\theta = 2\beta)$)。上と同じエンコーディングを再現するため、2 で割っています。


In [49]:
from qiskit.circuit.library import z_feature_map

zfeature_map = z_feature_map(feature_dimension=2, reps=3)
zfeature_map = zfeature_map.assign_parameters([(1 / 2) * pi / 2, (1 / 2) * pi / 3])
zfeature_map.decompose().draw("mpl")

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/c15b8fe2-ae83-4c76-a908-71596deb7d82-0.avif" alt="Output of the previous code cell" />

明らかに、これは上で手作業で行ったものとは異なるマッピングですが、パラメーターの順序が一貫していることに注目してください。$\textstyle\frac{1}{2}\pi$ は今回も $0^\text{th}$ 番目の量子ビットにエンコードされています。

ZFM は Qiskit の ZFM クラスを通じて利用できます。また、この構造を参考にして独自の特徴量マッピングを構築することもできます。


### $ZZ$ 特徴量マップ

$ZZ$ 特徴量マップ (ZZFM) は、2 量子ビットのエンタングルゲート、具体的には $ZZ$ 回転ゲート $R_{ZZ}(\theta)$ を加えることで ZFM を拡張したものです。ZZFM は、ZFM とは異なり、一般に古典コンピューターでの計算コストが高いと予想されています。

$R_{ZZ}(\theta)$ は $ZZ$ 相互作用を実装するもので、$\theta = \textstyle{\frac{1}{2}}\pi$ のときに最大限にエンタングルさせます。$R_{ZZ}(\theta)$ は、[RZZ ゲート](/docs/api/qiskit/qiskit.circuit.library.RZZGate) と `QuantumCircuit` クラスのメソッド `decompose` を使った次の Qiskit コードで示すように、2 量子ビット上の一連のゲートに分解できます。データベクトル $\vec{x}$ の単一の特徴量 $\vec{x}_k=\pi$ をエンコードします。


In [50]:
qc = QuantumCircuit(2)
qc.rzz(pi, 0, 1)
qc.draw("mpl", scale=1)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/6c312a5f-91a5-499c-a391-efc73cd0e4e1-0.avif" alt="Output of the previous code cell" />

よくあることですが、.decompose() を使って構成ゲートをすべて表示するまでは、これは単一のゲートのようなユニットとして表示されます。


In [51]:
qc.decompose().draw("mpl", scale=1)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/92062646-78f2-4dd6-82ad-7a543cb6a566-0.avif" alt="Output of the previous code cell" />

データは 2 番目の量子ビット上の位相回転 $P(\theta) = e^{i\theta/2}R_Z(\theta)$ によってマッピングされます。$R_{ZZ}(\theta)$ ゲートは、作用する 2 つの量子ビットを、エンコードされた特徴量の値によって決まる度合いでエンタングルさせます。

ZZFM 回路全体は、ZFM と同様のアダマールゲートと位相ゲート、およびそれに続く上述のエンタングルメントで構成されます。ZZFM 回路の 1 回の繰り返しは次のとおりです。

$$
\mathscr{U}_{\text{ZZFM}}(\vec{x}) = U_{ZZ}(\vec{x})\big(P(\vec{x}_1)\otimes\ldots P(\vec{x}_k)\otimes\ldots P(\vec{x}_N)H^{\otimes N}\big)=U_{ZZ}(\vec{x})\left(\bigotimes_{k = 1}^N P(\vec{x}_k)\right)H^{\otimes N},
$$

ここで、$U_{ZZ}(\vec{x})$ には、エンタングルメント方式によって構造化された ZZ ゲート層が含まれます。いくつかのエンタングルメント方式を以下のコードブロックで示します。$U_{ZZ}(\vec{x})$ の構造には、エンタングルされる量子ビットのデータ特徴量を次のように組み合わせる関数も含まれます。$R_{ZZ}$ ゲートを量子ビット $p$ と $q$ に適用するとしましょう。位相層では、これらの量子ビットにはそれぞれ $\vec{x}_p$ と $\vec{x}_q$ をエンコードする位相ゲートがあります。$R_{ZZ,q,p}(\theta_{q,p})$ の引数 $\theta_{q,p}$ は、単にこれらの特徴量のどちらか一方になるのではなく、多くの場合 $\phi$ で表される関数 (方位角と混同しないでください) になります。

$$
\theta_{q,p} \rightarrow \phi(\vec{x}_q, \vec{x}_p) = 2(\pi-\vec{x}_q)(\pi-\vec{x}_p).
$$

これについては、以下のいくつかの例で確認します。複数回の繰り返しへの拡張は、`z_feature_map` の場合と同じです。

$$
\mathscr{U}^{(r)}_{\text{ZZFM}}\left(\vec{x}\right)=\prod_{s=1}^{r}\left[U_{ZZ}(\vec{x})\left(\bigotimes_{k = 1}^N P(\vec{x}_k)\right)H^{\otimes N}\right].
$$

演算子が複雑になってきたので、まず次のコードを使って、データベクトル $\vec{x} = (x_0, x_1)$ を 2 量子ビット、繰り返し 1 回の ZZFM でエンコードしてみましょう。


In [52]:
from qiskit.circuit.library import zz_feature_map

feature_dim = 2
zzfeature_map = zz_feature_map(
    feature_dimension=feature_dim, entanglement="linear", reps=1
)
zzfeature_map.decompose(reps=1).draw("mpl", scale=1)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/1ec2f2e1-b665-4dde-a223-35489f17c695-0.avif" alt="Output of the previous code cell" />

Qiskit のデフォルトでは、特徴量 $(\vec{x}_1, \vec{x}_2)$ はマッピング関数 $\theta_{1,2} = \phi(\vec{x}_1, \vec{x}_2) = 2(\pi-\vec{x}_1)(\pi-\vec{x}_2)$ によってまとめて $R_{ZZ}(\theta)$ にマッピングされます。Qiskit では、前処理のステップとして、関数 $\phi$ (または $\phi_S$。ここで $S$ は $R_{ZZ}$ ゲートを通じて結合される量子ビットのペアの集合) をユーザーがカスタマイズできます。

4 次元のデータベクトル $\vec{x} = (\vec{x}_1, \vec{x}_2, \vec{x}_3, \vec{x}_4)$ に移り、繰り返し 1 回の 4 量子ビット ZZFM にマッピングすると、さまざまな量子ビットのペアに対するマッピング $\phi$ が見えてきます。また、「linear (線形)」エンタングルメントの意味もわかります。


In [53]:
feature_dim = 4
zzfeature_map = zz_feature_map(
    feature_dimension=feature_dim, entanglement="linear", reps=1
)
zzfeature_map.decompose().draw("mpl", scale=1)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/979c765a-e3d8-4e3e-8e52-3f816203a934-0.avif" alt="Output of the previous code cell" />

線形エンタングルメント方式では、この回路内で (番号が) 隣り合う量子ビットのペアがエンタングルされます。Qiskit には、`circular` や `full` など、他の組み込みのエンタングルメント方式もあります。


### パウリ特徴量マップ

パウリ特徴量マップ (PFM) は、ZFM と ZZFM を任意のパウリゲートを使えるように一般化したものです。パウリ特徴量マップは、前の 2 つの特徴量マップと非常によく似た形をしています。ベクトル $\vec{x}$ の $N$ 個の特徴量のエンコーディングを $r$ 回繰り返す場合、次のようになります。

$$
\mathscr{U}_{\text{PFM}}(\vec{x}) = \prod_{s=1}^{r} U(\vec{x}) H^{\otimes n}.
$$

PFM では、$U(\vec{x})$ はパウリ展開ユニタリー演算子に一般化されます。ここでは、これまでに扱った特徴量マップのより一般化された形を示します。

$$
U(\vec{x}) = \exp\left(i \sum_{S \in\mathcal{I}} \phi_S(\vec{x}) \prod_{i \in S} \sigma_i \right),
$$

ここで、$\sigma_i$ はパウリ演算子 $\sigma_i \in {I,X,Y,Z}$ です。また、$\mathcal{I}$ は特徴量マップによって決まるすべての量子ビット結合の集合であり、単一量子ビットゲートが作用する量子ビットの集合も含みます。つまり、量子ビット 0 に位相ゲートが作用し、量子ビット 2 と 3 に $R_{ZZ}$ ゲートが作用する特徴量マップの場合、集合 $\mathcal{I}$ には $\{\{0\},\{2,3\}\}$ が含まれます。$S$ はその集合のすべての要素にわたって動きます。これまでの特徴量マップでは、関数 $\phi_S(\vec{x})$ は単一量子ビットゲートのみ、または 2 量子ビットゲートのみに関係していました。ここでは、これを一般的に定義します。

$$
\phi_S(\vec{x})=
    \begin{cases}
      x_i & \text{if } S= \{i\} \text{ (single-qubit)}\\
      \prod_{j\in{S}}(\pi-x_j) & \text{if } |S|\ge2 \text{ (multi-qubit)}\\
    \end{cases}
$$

ドキュメントについては、[Qiskit `Pauli feature map` クラスのドキュメント](/docs/api/qiskit/qiskit.circuit.library.PauliFeatureMap)) を参照してください。ZZFM では、演算子 $\sigma_i$ は $Z_i$ に限定されます。

上記のユニタリーを理解する方法の 1 つは、物理系におけるプロパゲーターとの類推です。上記のユニタリーは、イジング模型に似たハミルトニアン $\mathcal{H}$ に対するユニタリー時間発展演算子 $\exp(it\mathcal{H})$ であり、時間パラメーター $t$ が、時間発展を駆動するデータ値に置き換えられたものです。このユニタリー演算子を展開すると PFM 回路が得られます。$S$ におけるエンタングルの結合は、スピン格子におけるイジング結合として解釈できます。


これらのイジング型相互作用を表すパウリ $Y$ 演算子と $XX$ 演算子の例を考えてみましょう。Qiskit には、単一量子ビットゲートと $n$ 量子ビットゲートを選択して PFM をインスタンス化するための `pauli_feature_map` クラスが用意されています。この例では、それらをパウリ文字列 `‘Y’` と `‘XX’` として渡します。通常、$n$ は単一量子ビット相互作用の場合は 1、2 量子ビット相互作用の場合は 2 です。エンタングルメント方式は「linear (線形)」であり、量子回路内で隣り合う量子ビットのみが結合されることを意味します。この量子回路は抽象化レイヤーであるため、これは量子コンピューター自体の上で隣り合う量子ビットに対応するわけではないことに注意してください。


In [54]:
from qiskit.circuit.library import pauli_feature_map

feature_dim = 3
pfmap = pauli_feature_map(
    feature_dimension=feature_dim, entanglement="linear", reps=1, paulis=["Y", "XX"]
)

pfmap.decompose().draw("mpl", scale=1.5)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/5ba5df82-83c1-428c-a269-baea5f75c3dc-0.avif" alt="Output of the previous code cell" />

Qiskit では、パウリ回転のスケーリングを制御するためのパラメーター $\alpha$ がパウリ特徴量マップに用意されています。

$$
U(\bar{x}) = \exp\left(i \alpha \sum_{S\subseteq[n]} \phi_S(\bar{x}) \prod_{i \in S} \sigma_i \right)
$$

$\alpha$ のデフォルト値は $2$ です。その値をたとえば区間 $[0,4]$ で最適化することで、量子カーネルをデータによりよく適合させることができます。


### パウリ特徴量マップのギャラリー

ここでは、2 量子ビット回路のさまざまなパウリ特徴量マップを可視化して、どのような可能性があるのかをより具体的に把握します。


In [55]:
from qiskit.visualization import circuit_drawer
import matplotlib.pyplot as plt

feature_dim = 2
fig, axs = plt.subplots(9, 2)
i_plot = 0
for paulis in [
    ["I"],
    ["X"],
    ["Y"],
    ["Z"],
    ["XX"],
    ["XY"],
    ["XZ"],
    ["YY"],
    ["YZ"],
    ["ZZ"],
    ["X", "ZZ"],
    ["Y", "ZZ"],
    ["Z", "ZZ"],
    ["X", "YZ"],
    ["Y", "YZ"],
    ["Z", "YZ"],
    ["YY", "ZZ"],
    ["XY", "ZZ"],
]:
    pfmap = pauli_feature_map(feature_dimension=feature_dim, paulis=paulis, reps=1)
    circuit_drawer(
        pfmap.decompose(),
        output="mpl",
        style={"backgroundcolor": "#EEEEEE"},
        ax=axs[int((i_plot - i_plot % 2) / 2), i_plot % 2],
    )
    axs[int((i_plot - i_plot % 2) / 2), i_plot % 2].title.set_text(paulis)
    i_plot += 1

fig.set_figheight(16)
fig.set_figwidth(16)

<Image src="/learning/images/courses/quantum-machine-learning/data-encoding/extracted-outputs/69225757-193a-490b-8ca4-d1b187b774b3-0.avif" alt="Output of the previous code cell" />

もちろん、上記を拡張して、パウリ行列の他の順列や繰り返しを含めることもできます。学習者の皆さんは、これらのオプションをぜひ試してみてください。


## 組み込み特徴量マップのまとめ

ここまでで、データを量子回路にエンコードするためのいくつかの方式を見てきました。

* 基底エンコーディング
* 振幅エンコーディング
* 角度エンコーディング
* 位相エンコーディング
* 密エンコーディング

これらのエンコーディング方式を使って独自の特徴量マップを構築する方法を学び、角度エンコーディングと位相エンコーディングを活用する 4 つの組み込み特徴量マップも見てきました。

* Efficient SU2
* Z 特徴量マップ
* ZZ 特徴量マップ
* パウリ特徴量マップ

これらの組み込み特徴量マップは、いくつかの点で互いに異なっていました。

* 与えられた数の特徴量をエンコードする際の深さ
* 与えられた数の特徴量に必要な量子ビット数
* エンタングルメントの度合い (当然ながら他の違いと関連しています)

以下のコードでは、これら 4 つの組み込み特徴量マップを特徴量セットのエンコーディングに適用し、得られた回路の 2 量子ビットゲートの深さをプロットします。2 量子ビットゲートのエラー率は単一量子ビットゲートのエラー率よりもはるかに高いため、2 量子ビットゲートの深さに最も関心を持つのは妥当でしょう。以下のコードでは、まず回路を分解し、次に count\_ops() を使うことで、回路内のすべてのゲートの数を取得しています。ここで注目する 2 量子ビットゲートは 'cx' ゲートです。


In [ ]:
# パラメーターと深さ用の空のリストを初期化
x = [0.1, 0.2]
n_data = []
zz2gates = []
su22gates = []
z2gates = []
p2gates = []

# 特徴量マップを生成
for n in range(3, 10):
    x.append(n / 10)
    zzcircuit = zz_feature_map(n, reps=1, insert_barriers=True)
    zcircuit = z_feature_map(n, reps=1, insert_barriers=True)
    su2circuit = efficient_su2(n, reps=1, insert_barriers=True)
    pcircuit = pauli_feature_map(n, reps=1, paulis=["XX"], insert_barriers=True)
    # cx の深さを取得
    zzcx = zzcircuit.decompose().count_ops().get("cx")
    zcx = zcircuit.decompose().count_ops().get("cx")
    su2cx = su2circuit.decompose().count_ops().get("cx")
    pcx = pcircuit.decompose().count_ops().get("cx")

    # cx ゲート数をリストに追加します。zz とパウリのデータ点は重なるため、
    # 位置をずらしています。
    n_data.append(n)
    zz2gates.append(zzcx - 0.5)
    z2gates.append(0)
    su22gates.append(su2cx)
    p2gates.append(pcx + 0.5)

# 出力をプロット
plt.plot(n_data, p2gates, "bo")
plt.plot(n_data, zz2gates, "ro")
plt.plot(n_data, su22gates, "yo")
plt.plot(n_data, z2gates, "go")
plt.ylabel("CX Gates")
plt.xlabel("Data elements")
plt.legend(["Pauli", "ZZ", "SU2", "Z"])
# plt.suptitle('zz_feature_map(n)')
plt.show()

一般に、パウリ特徴量マップと ZZ 特徴量マップは、`efficient_su2` や Z 特徴量マップよりも回路の深さが大きくなり、2 量子ビットゲートの数も多くなります。

Qiskit に組み込まれている特徴量マップは幅広く適用できるため、特に学習段階では、独自の特徴量マップを設計する必要がないことも多いでしょう。しかし、量子機械学習の専門家は、次の 2 つの複雑な課題に取り組む中で、独自の特徴量マッピングの設計というテーマに立ち返ることになるでしょう。

1. 現代のハードウェア: ノイズの存在や誤り訂正符号の大きなオーバーヘッドのため、現在のアプリケーションでは、ハードウェア効率や 2 量子ビットゲートの深さの最小化などを考慮する必要があります。

2. 目の前の問題に適合するマッピング: たとえば `zz_feature_map` は古典的にシミュレーションするのが難しく、したがって興味深いと言うことと、`zz_feature_map` が**あなたの**機械学習タスクやデータセットに理想的に適していることとは、まったく別の話です。さまざまな種類のデータに対するさまざまなパラメーター化量子回路の性能は、活発に研究が進められている分野です。

最後に、ハードウェア効率について触れておきます。


## ハードウェア効率の良い特徴量マッピング

ハードウェア効率の良い特徴量マッピングとは、計算におけるノイズやエラーを減らすために、実際の量子コンピューターの制約を考慮した特徴量マッピングです。近い将来の量子コンピューターで量子回路を実行する場合、ハードウェアに固有のノイズを軽減するための戦略が数多くあります。ハードウェア効率のための主な戦略の 1 つは、量子回路の深さを最小化して、ノイズやデコヒーレンスが計算を損なう時間を短くすることです。量子回路の深さとは、(回路最適化後に) 計算全体を完了するのに必要な、時間的に揃えられたゲートのステップ数です[\[5\]](#references)。抽象的な論理回路の深さは、実際の量子コンピューター向けにトランスパイルされた後の回路の深さよりもはるかに小さい場合があることを思い出してください。

トランスパイルとは、ハードウェアの制約を考慮しながら、量子回路を高レベルの抽象表現から実際の量子コンピューターで実行できる形式に変換するプロセスです。量子コンピューターには、ネイティブな単一量子ビットゲートと 2 量子ビットゲートのセットがあります。つまり、Qiskit コード内のすべてのゲートを、ネイティブなハードウェアゲートのセットにトランスパイルする必要があります。たとえば、2023 年に完成した Heron r1 プロセッサーを搭載する QPU である ibm\_torino では、ネイティブゲート (基底ゲート) は `{CZ, ID, RZ, SX, X}` です。これらはそれぞれ、2 量子ビットの制御 Z ゲートと、恒等ゲート、$Z$ 回転、NOT の平方根、NOT と呼ばれる単一量子ビットゲートであり、ユニバーサルなゲートセットを構成します。多量子ビットゲートを等価なサブ回路として実装する場合、ハードウェアで利用可能な他の単一量子ビットゲートとともに、物理的な 2 量子ビット $CZ$ ゲートが必要になります。さらに、物理的に結合されていない量子ビットのペアに 2 量子ビットゲートを実行するには、結合を可能にするために量子ビット間で量子ビットの状態を移動させる SWAP ゲートが追加されるため、回路が長くなることは避けられません。`optimization` 引数は 0 から最高レベルの 3 まで設定できます。より細かく制御・カスタマイズするには、[Qiskit Pass Manager](/docs/api/qiskit/qiskit.transpiler.PassManager) でトランスパイラーのパイプラインを管理できます。トランスパイルの詳細については、[Qiskit Transpiler のドキュメント](/docs/api/qiskit/transpiler) を参照してください。

Havlicek et al. 2019 [\[2\]](#references) では、著者らがハードウェア効率を実現する方法の 1 つとして、$ZZ$ 特徴量マップを使用しています。これは 2 次の展開だからです (上記の「$ZZ$ 特徴量マップ」のセクションを参照してください)。$N$ 次の展開には $N$ 量子ビットゲートが含まれます。IBM® の量子コンピューターには $N>2$ のネイティブな $N$ 量子ビットゲートがないため、それらを実装するには、ハードウェアで利用可能な 2 量子ビットの CNOT ゲートに分解する必要があります。著者らが深さを最小化する 2 つ目の方法は、アーキテクチャーの結合に直接マッピングされる $ZZ$ 結合トポロジーを選択することです。さらに著者らが行っている最適化は、より高性能で適切に接続されたハードウェアのサブ回路をターゲットにすることです。その他に考慮すべき点としては、特徴量マップの繰り返し回数を最小化することや、すべての量子ビットをエンタングルさせる「full」方式の代わりに、カスタマイズされた低深度の方式や「linear (線形)」のエンタングルメント方式を選択することが挙げられます。

![データエンコーディングの画像](https://quantum.cloud.ibm.com/learning/images/courses/quantum-machine-learning/data-encoding/qml-03-data-encoding-24.avif)

上の図は、物理量子ビットとハードウェアの結合をそれぞれ表すノードとエッジのネットワークを示しています。ibm\_torino の結合マップと性能が、可能なすべての 2 量子ビット CZ 結合ゲートとともに示されています。量子ビットは、マイクロ秒 (μs) 単位の T1 緩和時間に基づくスケールで色分けされており、T1 時間が長いほど良く、より明るい色調で表示されます。結合エッジは CZ エラーによって色分けされており、色が濃いほど良いことを示します。ハードウェア仕様に関する情報は、ハードウェアのバックエンド構成スキーマ `IBMQBackend.configuration()` で確認できます。


## 参考文献

1. Maria Schuld and Francesco Petruccione, *Supervised Learning with Quantum Computers*, Springer 2018, [doi:10.1007/978-3-319-96424-9](https://www.springer.com/gp/book/9783319964232).
2. <a id="Havlicek2018" />Vojtech Havlicek et al., “Supervised Learning with Quantum Enhanced Feature Spaces.” *Nature*, vol. 567 (2019): 209–212. [https://arxiv.org/abs/1804.11326](https://arxiv.org/abs/1804.11326).
3. Ryan LaRose and Brian Coyle, "Robust data encodings for quantum classifiers", Physical Review A 102, 032420 (2020), [doi:10.1103/PhysRevA.102.032420](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.102.032420), [arXiv:2003.01695](https://arxiv.org/abs/2003.01695).
4. <a id="Grover2002" />Lou Grover and Terry Rudolph. “Creating Superpositions That Correspond to Efficiently Integrable Probability Distributions.” arXiv:quant-ph/0208112, August 15, 2002, [https://arxiv.org/abs/quant-ph/0208112](https://arxiv.org/abs/quant-ph/0208112).
5. Adrián Pérez-Salinas, Alba Cervera-Lierta, Elies Gil-Fuster, José I. Latorre, "Data re-uploading for a universal quantum classifier",  [Quantum 4, 226 (2020)](https://quantum-journal.org/papers/q-2020-02-06-226/), [ArXiv.org/abs/1907.02085](https://arxiv.org/abs/1907.02085).
6. Maria Schuld, Ryan Sweke, Johannes Jakob Meyer, "The effect of data encoding on the expressive power of variational quantum machine learning models", [Phys. Rev. A 103, 032430 (2021)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.103.032430), [arxiv.org/abs/2008.08605](https://arxiv.org/abs/2008.08605)


In [ ]:
import qiskit

qiskit.version.get_version_info()

© IBM Corp., 2017-2026